# NTN RT Sensitivity Sweeps

Driver for the sensitivity analysis of the RT → TR 38.811 LSP pipeline.

* **Every sampled parameter is uniform**: continuous factors are U[lo, hi], integers and
  categories are discrete-uniform. Draws are Latin-hypercube, one per scene.
* **Scenes are sampled too**: each scene has its own P.1410 (α, β, γ) and street-grid heading.
* **Real orbits**: every (scene, elevation) gets a real Starlink pass (cached TLE catalog,
  SGP4) whose peak matches a sampled peak elevation, at an area-uniform ground location.
* **No scene caching**: manifests only *describe* scenes. Each runner subprocess builds a
  scene in scratch, solves it, deletes it. This kernel never imports Mitsuba or Sionna.

Variants (material, link, solver) reuse the same scenes, orbits, UEs and solver seed as the
baseline, so every effect is paired. Analysis: `../analysis/sensitivity_analysis.ipynb`.
Details: `docs/sensitivity.md`.

## 1 - Environment and imports

In [1]:
import os, sys
from pathlib import Path

os.environ['DRJIT_LIBLLVM_PATH'] = '/usr/lib/x86_64-linux-gnu/libLLVM-20.so'

# Modules import as Ray_Tracing.src.*: put the repo's parent on sys.path. The repo root
# is the nearest ancestor holding src/utils, searched from this notebook's own folder
# (VS Code sets __vsc_ipynb_file__), then from the kernel's working directory, which
# may also be the workspace root one level above Ray_Tracing/.
_starts = [Path(p).resolve().parent for p in [globals().get("__vsc_ipynb_file__")] if p]
_starts.append(Path.cwd().resolve())
REPO = next((d for s in _starts for d in (s, *s.parents, s / "Ray_Tracing")
             if (d / "src" / "utils").is_dir()), None)
if REPO is None:
    raise RuntimeError(f"Ray_Tracing repo not found above {[str(s) for s in _starts]}")
if str(REPO.parent) not in sys.path:
    sys.path.insert(0, str(REPO.parent))
print("repo:", REPO)

import copy, json, pickle, subprocess
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from Ray_Tracing.src.utils import scene_gen_ppp as sg      # parameter defaults only; no Mitsuba here
from Ray_Tracing.src.utils import sweep_utils as su
from Ray_Tracing.src.utils import variant_manifest as vm
from Ray_Tracing.src.utils.checkpoint_utils import load_results

repo: /path/to/Ray_Tracing


## 2 - Baseline run settings

Fixed for every variant. Changing anything here invalidates finished variants (delete
`SWEEP_ROOT` or change it). `SOLVER_SEED_OFFSET` fixes the ray sample per (scene, elevation),
so variants are compared on identical rays (identical to ~1e-5).

In [2]:
SWEEP_ROOT    = str(REPO / "Results" / "sensitivity")
GPUS          = (0, 1)
ELEVATIONS    = [10, 20, 30, 40, 50, 60, 70, 80, 90]   # baseline (Phase 0)
SUB_ELEVS     = [10, 20, 30, 40, 50, 60, 70, 80, 90]                       # replicates and Phases 1-3
FREQS         = {"2GHz": 2e9, "20GHz": 20e9}

N_SCENES       = 100     # base ensemble (Phases 0-3)
N_RX_PER_SCENE = 100

# PPP generator. cell_res / alpha / gamma are overwritten per scene by the scene draws (§3);
# points_per_cell=4 keeps realised alpha unbiased up to 0.5, h_max=120 avoids clipping heights.
PPP_KW = dict(Lx=500.0, Ly=500.0, cell_res=40.0, alpha=0.30, points_per_cell=4,
              gamma=15.0, h_min=4.0, h_max=120.0, max_fill=0.8, min_side=5.0)
PPP_MATERIALS = dict(building_material="itu_concrete", roof_material="itu_brick",
                     ground_material="itu_plasterboard")   # names are dummy identities
S0 = 0.4                                                  # baseline scattering coefficient
ROLE_PARAMS = {r: dict(p, scattering_coefficient=S0) for r, p in
               (("wall", sg.DEFAULT_WALL_PARAMS), ("roof", sg.DEFAULT_ROOF_PARAMS),
                ("ground", sg.DEFAULT_GROUND_PARAMS))}

BASE_CFG = {
    "FREQUENCY": 2e9, "TX_POWER_DBM": 74.0, "SLANT_DIST_M": 600e3,   # fallback only: orbits set TX
    "SCENE_CENTER": [0.0, 0.0], "RX_HEIGHT": 1.5,
    "SCATTERING_COEFF": S0, "MAX_DEPTH": 3, "LOS": True, "SPECULAR_REFLECTION": True,
    "DIFFUSE_REFLECTION": True, "DIFFRACTION": True, "EDGE_DIFFRACTION": False,
    "REFRACTION": False, "BEAMFORMING": True,
    "SCENE_RADIUS": float(np.hypot(PPP_KW["Lx"] / 2, PPP_KW["Ly"] / 2)),
    "BASE_SAMPLES_PER_SRC": 10_000_000, "BASE_MAX_PATHS": 10_000_000,   # 1e6 rays is ~3x faster; decide from phase0's noise floor
    "MERGE_SHAPES": True,        # 3 shapes, not ~400: solve 0.7 s instead of ~22 s of Dr.Jit codegen
    "RX_CHUNK": 25,              # UEs per solve: the 1e7 candidate cap / dedupe table is shared by a solve's UEs
    "RNG_SEED": 42, "CHECKPOINT_INTERVAL": 10, "RAW_SAVE_INTERVAL": 200,
    "N_RX_PER_SCENE": N_RX_PER_SCENE, "PPP_RX_GRID_RES": 2.0, "PPP_RX_MARGIN": 20.0,
    "PPP_RX_CLEARANCE": 1.0, "PPP_VERIFY_EVERY": 0,
    "TX_POL_TYPE": "RHCP", "RX_POL_TYPE": "RHCP", "POL_BASE_PATTERN": "iso",
    # Ionosphere (P.531: per-scene log-normal VTEC -> Faraday roll before the trace, group
    # delay / phase after) and troposphere (P.676 gas, P.838 rain, P.618 scintillation after
    # the trace). Draws are seeded per (scene, elevation), so variants and carriers pair.
    # Uniform losses leave K / DS / angular spreads unchanged; they show in PG_dB.
    "IONOSPHERE_ENABLED": True, "VTEC_MEAN_TECU": 10.0, "VTEC_STD_TECU": 5.0, "B_L_TESLA": 2e-5,
    "ATMOSPHERE_ENABLED": True, "ATM_RHO_G_M3": 7.5, "RAIN_R_MM_H": 0.0, "ATM_SCINTILLATION": True,
    # Path-domain Doppler from each scene's orbit velocity: mean shift and power-weighted RMS
    # spread over paths. Free. The time-domain estimator (DOPPLER_NUM_TIME_STEPS > 1) cannot
    # resolve sub-Hz spreads (bin = 10 x max Doppler / steps ~ kHz) and costs ~4x -- keep it off.
    "DOPPLER_ENABLED": True, "DOPPLER_NUM_TIME_STEPS": 1,
    "SOLVER_SEED_OFFSET": 0, "K_FIT_N": 0,   # moment-fit K off: mostly NaN, ~1.3 s/UE
}

## 3 - Sampled parameters (all uniform)

`(lo, hi)` is U[lo, hi]; `(lo, hi, "int")` is discrete-uniform on lo..hi (equal mass per
integer). Factor names route by prefix: `geo.` scene sampler, `cfg.` per-scene config,
`<role>.` / `all.` material (`eps_scale`, `sigma_scale` multiply P.2040 `a`, `c`).
`BASELINE` is each factor's value in the base ensemble: the point every one-at-a-time
response is measured from.

The orbit draws live in `orbit_sampling` (§4): ground point area-uniform over |lat| ≤ `LAT_MAX`,
peak elevation U[max(30°, E + 1°), 90°] per (scene, E), leg discrete-uniform {rising, setting}.

In [3]:
SCENE_FACTORS = {                         # drawn for every scene, every phase (ITU-R P.1410)
    "geo.alpha": (0.10, 0.50),            # built-up fraction: suburban 0.1 .. dense urban 0.5
    "geo.beta": (300.0, 750.0),           # buildings / km^2 -> cell_res
    "geo.gamma": (8.0, 20.0),             # Rayleigh height scale [m]
    "cfg.SCENE_EAST_HEADING_DEG": (0.0, 90.0),   # street-grid orientation vs north
}
MATERIAL_FACTORS = {
    **{f"{r}.eps_scale": (0.5, 1.5) for r in ("ground", "wall", "roof")},
    **{f"{r}.sigma_scale": (0.5, 1.5) for r in ("ground", "wall", "roof")},
    **{f"{r}.scattering_coefficient": (0.0, 0.6) for r in ("ground", "wall", "roof")},
    "all.alpha_r": (1, 10, "int"),
    "all.xpd_coefficient": (0.0, 0.5),
}
# Orientation needs the scene_utils patch AND the Sionna RX-pattern fix (docs/sionna-patches.md).
ORIENTATION_READY = False
LINK_FACTORS = {"cfg.RX_HEIGHT": (1.0, 10.0),
                **({"cfg.RX_PITCH_DEG": (0.0, 90.0), "cfg.RX_YAW_DEG": (0.0, 360.0)}
                   if ORIENTATION_READY else {})}
ENV_FACTORS = {                          # swept in phase2_env; baseline values in BASE_CFG
    "cfg.VTEC_MEAN_TECU": (2.0, 60.0),   # TECU: quiet night .. mid-latitude daytime high
    "cfg.RAIN_R_MM_H": (0.0, 50.0),      # mm/h: clear .. heavy rain (P.837 R0.01 temperate-tropical)
    "cfg.ATM_RHO_G_M3": (3.0, 20.0),     # g/m^3 surface water vapour: dry .. tropical (P.836)
}
BASELINE = {**{k: 1.0 for k in MATERIAL_FACTORS if k.endswith("_scale")},
            **{k: S0 for k in MATERIAL_FACTORS if k.endswith("scattering_coefficient")},
            "all.alpha_r": 1, "all.xpd_coefficient": 0.0, "cfg.RX_HEIGHT": 1.5,
            "cfg.RX_PITCH_DEG": 90.0, "cfg.RX_YAW_DEG": 0.0,
            "cfg.VTEC_MEAN_TECU": 10.0, "cfg.RAIN_R_MM_H": 0.0, "cfg.ATM_RHO_G_M3": 7.5}
LAT_MAX, ORBIT_TOL_DEG, ORBIT_PROCS = 60.0, 1.0, 32
CATALOG = str(REPO / "tle_catalog" / "starlink.tle")

for name, spec in {**SCENE_FACTORS, **MATERIAL_FACTORS, **LINK_FACTORS, **ENV_FACTORS}.items():
    kind = "discrete-uniform" if spec[2:] == ("int",) else "uniform"
    print(f"{name:34s} {kind:17s} [{spec[0]}, {spec[1]}]   baseline {BASELINE.get(name, 'per-scene draw')}")

geo.alpha                          uniform           [0.1, 0.5]   baseline per-scene draw
geo.beta                           uniform           [300.0, 750.0]   baseline per-scene draw
geo.gamma                          uniform           [8.0, 20.0]   baseline per-scene draw
cfg.SCENE_EAST_HEADING_DEG         uniform           [0.0, 90.0]   baseline per-scene draw
ground.eps_scale                   uniform           [0.5, 1.5]   baseline 1.0
wall.eps_scale                     uniform           [0.5, 1.5]   baseline 1.0
roof.eps_scale                     uniform           [0.5, 1.5]   baseline 1.0
ground.sigma_scale                 uniform           [0.5, 1.5]   baseline 1.0
wall.sigma_scale                   uniform           [0.5, 1.5]   baseline 1.0
roof.sigma_scale                   uniform           [0.5, 1.5]   baseline 1.0
ground.scattering_coefficient      uniform           [0.0, 0.6]   baseline 0.4
wall.scattering_coefficient        uniform           [0.0, 0.6]   baseline 0.4
ro

## 4 - Base ensembles: sampled scenes + real-TLE orbit tables

`make_base` builds a file-free manifest: `N` scene seeds, one LHS draw of `SCENE_FACTORS`
each, and each scene's per-elevation real pass. The orbit table is built once by a CPU
subprocess pool (~10 s per scene / `ORBIT_PROCS`) and cached; it holds orbits, not scenes.

In [4]:
def make_base(n_scenes, base_seed, tag, scene_lhs_seed=0, orbit_seed=0):
    table_path = os.path.join(SWEEP_ROOT, f"orbit_table_{tag}.pkl")
    if not os.path.exists(table_path):
        subprocess.run([sys.executable, "-m", "Ray_Tracing.src.utils.orbit_sampling",
                        "--seeds", f"{base_seed}:{n_scenes}",
                        "--elevations", *[str(e) for e in ELEVATIONS],
                        "--out", table_path, "--catalog", CATALOG, "--procs", str(ORBIT_PROCS),
                        "--seed", str(orbit_seed), "--lat_max", str(LAT_MAX),
                        "--tol", str(ORBIT_TOL_DEG)],
                       cwd=str(REPO.parent), check=True)
    with open(table_path, "rb") as f:
        table = pickle.load(f)
    m = vm.virtual_manifest(n_scenes, PPP_KW, base_seed=base_seed, **PPP_MATERIALS,
                            wall_params=ROLE_PARAMS["wall"], roof_params=ROLE_PARAMS["roof"],
                            ground_params=ROLE_PARAMS["ground"])
    m = vm.lhs_manifest(m, SCENE_FACTORS, seed=scene_lhs_seed)
    return vm.attach_orbits(m, table)


BASE_MANIFEST = make_base(N_SCENES, 0, "base")
X = BASE_MANIFEST["design"]["X"]
print(f"{len(BASE_MANIFEST['scenes'])} scenes; design ranges:",
      {f: (round(X[:, i].min(), 2), round(X[:, i].max(), 2))
       for i, f in enumerate(BASE_MANIFEST["design"]["factors"])})
E_SHOW = float(SUB_ELEVS[len(SUB_ELEVS) // 2])
wp = [e["orbit"][E_SHOW] for e in BASE_MANIFEST["scenes"]]
print(f"orbits @{E_SHOW:g}°: {len({w['norad_id'] for w in wp})} distinct satellites, "
      f"slant {min(w['range_km'] for w in wp):.0f}-{max(w['range_km'] for w in wp):.0f} km, "
      f"catalog {BASE_MANIFEST['orbit_table']['catalog_sha256'][:12]}")

100 scenes; design ranges: {'geo.alpha': (0.1, 0.5), 'geo.beta': (302.62, 747.07), 'geo.gamma': (8.08, 19.95), 'cfg.SCENE_EAST_HEADING_DEG': (0.67, 89.29)}
orbits @50°: 100 distinct satellites, slant 423-732 km, catalog fa629af62f5d


## 5 - Variant definitions

One function per phase. Continuous factors in Phase 2 are **uniform one-at-a-time**: one
variant per factor, each scene draws that factor from its range, everything else at
`BASELINE`; the analysis fits the paired response Δ(x). Categorical choices are one
variant per category (discrete-uniform over categories).

> `alpha_r`, `scattering_pattern`, `RX_BASE_PATTERN` and orientation are applied by
> `scene_utils_sensitivity.py` (the patched fork of `scene_utils.py` the runner imports).
> Directional UE patterns (`cat_ue_*`, orientation) still need the Sionna RX-pattern fix
> (docs/sionna-patches.md); keep `ORIENTATION_READY = False` until it is archived.

In [5]:
# Sample cap for phases 1-2 (phase0, 3 and 4 keep their own designs): the first 50
# scenes, and per scene the first 50 of phase0's 100 UEs (25 at 10°, where a UE costs
# ~0.45 s against ~0.07 s at 20°+). Subsets of phase0's UEs, so OATs pair UE for UE.
CAP_P12      = {"N_RX_KEEP_BY_ELEV": {"10": 25, "*": 50}}
N_SCENES_P12 = 50


def per_freq(variants, extra_cfg=None, n_scenes=None):
    # Duplicate every variant at each carrier; FREQUENCY becomes part of the name.
    out = []
    for tag, f in FREQS.items():
        for v in variants:
            w = copy.deepcopy(v)
            w["name"] = f"{tag}__{v['name']}"
            w.setdefault("cfg_overrides", {}).update({**(extra_cfg or {}), "FREQUENCY": f})
            w.setdefault("elevations", ELEVATIONS)
            if n_scenes:
                w["n_scenes"] = n_scenes
            out.append(w)
    return out


def _oat(factors, elevations, seed0, extra_cfg=None):
    return [{"name": f"oat_{f}", "elevations": list(elevations),
             "oat": {"factor": f, "spec": spec, "seed": seed0 + i, "x0": BASELINE[f]},
             **({"cfg_overrides": dict(extra_cfg)} if extra_cfg else {})}
            for i, (f, spec) in enumerate(factors.items())]


def phase0(n_rep=5):
    return per_freq([{"name": "baseline"}] +
                    [{"name": f"rep{i}", "cfg_overrides": {"SOLVER_SEED_OFFSET": i},
                      "elevations": SUB_ELEVS} for i in range(1, n_rep)])


def phase1(depths=(1, 2, 3, 4, 5), elevations=SUB_ELEVS):
    return per_freq([{"name": f"depth{d}_diffuse{int(a)}_diffr{int(b)}", "elevations": list(elevations),
                      "cfg_overrides": {"MAX_DEPTH": d, "DIFFUSE_REFLECTION": a, "DIFFRACTION": b}}
                     for d in depths for a in (True, False) for b in (True, False)], CAP_P12, N_SCENES_P12)


def phase2_materials(elevations=SUB_ELEVS):
    vs = _oat(MATERIAL_FACTORS, elevations, seed0=1000)
    vs += [{"name": "cat_lambertian", "elevations": list(elevations),
            "material_overrides": {"all": {"scattering_pattern": "lambertian"}}},
           {"name": "check_thickness_0.1", "elevations": list(elevations),
            "material_overrides": {"all": {"thickness": 0.1}}}]
    return per_freq(vs, CAP_P12, N_SCENES_P12)


def phase2_link(elevations=SUB_ELEVS):
    vs = _oat({"cfg.RX_HEIGHT": LINK_FACTORS["cfg.RX_HEIGHT"]}, elevations, seed0=2000)
    if ORIENTATION_READY:
        vs += _oat({k: LINK_FACTORS[k] for k in ("cfg.RX_PITCH_DEG", "cfg.RX_YAW_DEG")},
                   elevations, seed0=2100, extra_cfg={"RX_BASE_PATTERN": "tr38901"})
    pols = {"RHCP_LHCP": ("RHCP", "LHCP"), "V_V": ("V", "V"), "H_H": ("H", "H"), "RHCP_V": ("RHCP", "V")}
    vs += [{"name": f"cat_pol_{k}", "elevations": list(elevations),
            "cfg_overrides": {"TX_POL_TYPE": t, "RX_POL_TYPE": r}} for k, (t, r) in pols.items()]
    vs += [{"name": f"cat_ue_{p}", "elevations": list(elevations),
            "cfg_overrides": {"RX_BASE_PATTERN": p}} for p in ("dipole", "tr38901")]
    return per_freq(vs, CAP_P12, N_SCENES_P12)


def phase2_env(elevations=SUB_ELEVS):
    # Rain / water vapour are applied after the trace and share one solve; VTEC
    # (Faraday roll of the TX) re-solves. Visible in PG_dB, and in K / DS only
    # through Faraday on linear polarization (CP is invariant to it).
    vs = _oat(ENV_FACTORS, elevations, seed0=3000)
    vs += [{"name": f"env_VV_vtec", "elevations": list(elevations),
            "oat": {"factor": "cfg.VTEC_MEAN_TECU", "spec": ENV_FACTORS["cfg.VTEC_MEAN_TECU"],
                    "seed": 3000, "x0": BASELINE["cfg.VTEC_MEAN_TECU"]},
            "cfg_overrides": {"TX_POL_TYPE": "V", "RX_POL_TYPE": "V"}}]
    return per_freq(vs, CAP_P12, N_SCENES_P12)


PHASE3_N_RX = 300


def phase3(elevations=SUB_ELEVS):
    # Same scenes and orbits, more UEs per scene: within- vs between-scene variance.
    return per_freq([{"name": "dense_ues", "elevations": list(elevations),
                     "cfg_overrides": {"N_RX_PER_SCENE": PHASE3_N_RX}}])


PHASE4_N_SCENES, PHASE4_N_RX = 1000, 20
PHASE4_FACTORS = {**MATERIAL_FACTORS, **LINK_FACTORS, **ENV_FACTORS}   # prune to factors that clear Phase 2


def phase4(elevation_bands=((10, 20), (60, 80))):
    path = os.path.join(SWEEP_ROOT, "phase4_manifest.pkl")
    if not os.path.exists(path):
        base = make_base(PHASE4_N_SCENES, 100_000, "lhs", scene_lhs_seed=1, orbit_seed=1)
        vm.lhs_manifest(base, PHASE4_FACTORS, seed=4, out_path=path)
    elevs = sorted({e for lo, hi in elevation_bands for e in ELEVATIONS if lo <= e <= hi})
    return per_freq([{"name": "lhs", "manifest": path, "elevations": elevs,
                      "cfg_overrides": {"N_RX_PER_SCENE": PHASE4_N_RX}}])


PHASES = {"phase0": phase0, "phase1": phase1, "phase2_materials": phase2_materials,
          "phase2_link": phase2_link, "phase2_env": phase2_env, "phase3": phase3, "phase4": phase4}


def scene_seconds(e, n_ue, depth=3):
    # Measured (RTX 3080, merged shapes, 1e7 rays, chunks of 25): ~0.7 s per scene plus
    # ~0.45 s per UE at 10° (≈1e6 paths per UE) or ~0.07 s per UE at 20°+; depth 5 ≈ 1.5x.
    return (0.7 + n_ue * (0.45 if e <= 10 else 0.07)) * (1 + 0.25 * max(depth - 3, 0))


def gpu_hours(variants):
    # One job per solve group (variants differing only in UE-side settings share each
    # scene's ray trace). A group's UEs: distinct UE geometries x 2 if it is dual-pol.
    groups = su.plan_groups(variants, BASE_CFG, BASE_MANIFEST)
    sec = 0.0
    for vs, e, n_sc in groups:
        c = {**BASE_CFG, **vs[0].get("cfg_overrides", {})}
        n_ue = int(c.get("N_RX_PER_SCENE", 100))
        kb = c.get("N_RX_KEEP_BY_ELEV") or {}
        n_ue = min(n_ue, int(kb.get(str(e), kb.get("*", n_ue))))
        geoms = 1 + sum(any(k in str(v.get("oat", {}).get("factor")) for k in ("HEIGHT", "PITCH", "YAW"))
                        for v in vs)
        pols = {(v.get("cfg_overrides", {}).get("TX_POL_TYPE"), v.get("cfg_overrides", {}).get("RX_POL_TYPE"))
                for v in vs}
        sec += n_sc * scene_seconds(e, n_ue * geoms * (2 if len(pols) > 1 else 1), int(c.get("MAX_DEPTH", 3)))
    return len(groups), sec / 3600


for k, f in PHASES.items():
    if k == "phase4" and not os.path.exists(os.path.join(SWEEP_ROOT, "orbit_table_lhs.pkl")):
        print(f"{k:18s} (builds a {PHASE4_N_SCENES}-scene orbit table on first call)")
        continue
    vs = f()
    n, h = gpu_hours(vs)
    print(f"{k:18s} {len(vs):3d} variants  {n:4d} jobs  ~{h:5.0f} GPU-h  "
          f"(~{h / len(GPUS) / 24:.1f} days on {len(GPUS)} GPUs)")

phase0              10 variants    90 jobs  ~   30 GPU-h  (~0.6 days on 2 GPUs)
phase1              40 variants   360 jobs  ~   29 GPU-h  (~0.6 days on 2 GPUs)
phase2_materials    26 variants   234 jobs  ~   16 GPU-h  (~0.3 days on 2 GPUs)
phase2_link         14 variants    54 jobs  ~    7 GPU-h  (~0.1 days on 2 GPUs)
phase2_env           8 variants    36 jobs  ~    4 GPU-h  (~0.1 days on 2 GPUs)
phase3               2 variants    18 jobs  ~   17 GPU-h  (~0.4 days on 2 GPUs)
phase4             (builds a 1000-scene orbit table on first call)


## 6 - Launch

One subprocess per free GPU over the jobs of every phase in `RUN`; variants that differ
only in UE-side settings share a job. Finished (variant, elevation) pairs are skipped;
interrupted ones resume from their checkpoint. With `DETACH`, the queue runs in the tmux
session `sweep`, so the notebook and SSH session can be closed; re-run this cell after a
crash or reboot to pick up where it stopped. §7 shows the last phase in `RUN`.

In [ ]:
RUN    = ["phase0"]    # phases to queue, run back to back, e.g. ["phase0", "phase1", "phase2_materials"]
RESUME = True
DETACH = True          # True: the queue runs in tmux and survives closing the notebook / SSH
BLOCK  = True          # DETACH=False only. False: returns once the last job is launched

entries = []
for PHASE in RUN:
    VARIANTS  = PHASES[PHASE]()
    SWEEP_DIR = os.path.join(SWEEP_ROOT, PHASE)
    n_jobs = sum(len(su.pending_elevations(SWEEP_DIR, v)) for v in VARIANTS)
    print(f"{PHASE}: {len(VARIANTS)} variants, {n_jobs} pending (variant, elevation) pairs -> {SWEEP_DIR}")
    for vs, e, _ in su.plan_groups(VARIANTS, BASE_CFG, BASE_MANIFEST):
        if len(vs) > 1:
            print(f"  elev={e}: one solve for {', '.join(v['name'] for v in vs)}")
    entries += su.queue_entries(SWEEP_DIR, VARIANTS, BASE_CFG, BASE_MANIFEST, resume=RESUME)

if DETACH:
    # Progress: `tail -f Results/sensitivity/_queue.log`, or `tmux attach -t sweep`.
    su.start_detached(entries, SWEEP_ROOT, gpus=GPUS, session="sweep")
else:
    active_procs = su.run_queue(entries, gpus=GPUS, block=BLOCK)


## 7 - Status and quick look

In [ ]:
su.sweep_status(SWEEP_DIR, VARIANTS)

In [ ]:
# K CDF (LoS UEs) per variant at one elevation; full analysis is in ../analysis/sensitivity_analysis.ipynb.
QL_ELEV = 30
fig, ax = plt.subplots(figsize=(7, 4))
for v in VARIANTS:
    if QL_ELEV not in v["elevations"]:
        continue
    r = load_results(su.variant_dir(SWEEP_DIR, v), QL_ELEV)
    if r is None:
        continue
    Pl, Pn = np.asarray(r["P_los"]), np.asarray(r["P_nlos"])
    m = (Pl > 0) & (Pn > 0)
    k = np.sort(10 * np.log10(Pl[m] / Pn[m]))
    ax.plot(k, np.linspace(0, 1, len(k)), label=v["name"], lw=1)
ax.set(xlabel="K [dB]", ylabel="CDF", title=f"{PHASE}, elevation {QL_ELEV}°")
ax.legend(fontsize=6, ncol=2)
plt.show()